In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/model_clean.csv")
y = "market_value_in_eur"

train, temp = train_test_split(df, test_size=0.30, random_state=42)
val, test = train_test_split(temp, test_size=0.50, random_state=42)

print("Trening:", len(train), " Validering:", len(val), " Test:", len(test))

Trening: 3773  Validering: 809  Test: 809


In [2]:
def evaluer(navn, y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rel = np.abs(y_pred - y_true) / y_true
    return {
        "modell": navn,
        "median_rel_feil": np.median(rel),
        "MAE_mill_eur": np.mean(np.abs(y_pred - y_true)) / 1e6,
    }

resultater = []

In [3]:
tabell = train.groupby(["league", "position"])[y].median().to_dict()
liga_median = train.groupby("league")[y].median().to_dict()
total_median = train[y].median()

def baseline_predict(data):
    return [
        tabell.get((l, p), liga_median.get(l, total_median))
        for l, p in zip(data["league"], data["position"])
    ]

pd.Series(tabell).unstack().round(-3)

,Attack,Defender,Goalkeeper,Midfield
BE1,2500000.0,2000000.0,900000.0,2350000.0
DK1,1200000.0,1200000.0,675000.0,800000.0
ES1,5000000.0,4000000.0,2900000.0,5000000.0
FR1,7000000.0,6000000.0,2500000.0,8000000.0
GB1,28000000.0,18000000.0,10000000.0,25000000.0
GR1,700000.0,500000.0,700000.0,850000.0
IT1,9000000.0,6000000.0,3250000.0,7000000.0
L1,8000000.0,5500000.0,4000000.0,8000000.0
NL1,1500000.0,1500000.0,2000000.0,1250000.0
PO1,1500000.0,1200000.0,1100000.0,2000000.0


In [4]:
pred_val = baseline_predict(val)
resultater.append(evaluer("Baseline (median liga × posisjon)", val[y], pred_val))

pd.DataFrame(resultater)

,modell,median_rel_feil,MAE_mill_eur
0,Baseline (median liga × posisjon),0.7,5.785878


In [5]:
kategorier = ["position", "sub_position", "foot", "league"]
numeriske = [
    "age", "height_in_cm", "contract_years_left", "contract_missing",
    "games", "minutes", "goals", "assists", "yellow",
    "goals_per90", "assists_per90", "cl_minutes",
    "international_caps", "international_goals",
]

X_train, y_train = train[kategorier + numeriske], train[y]
X_val, y_val = val[kategorier + numeriske], val[y]
X_test, y_test = test[kategorier + numeriske], test[y]

In [6]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeriske),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), kategorier),
])

def lag_modell(regressor):
    pipe = Pipeline([("prep", preprocess), ("model", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

In [7]:
modeller = {
    "Ridge": lag_modell(Ridge(alpha=1.0)),
    "Random Forest": lag_modell(RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    "HistGradientBoosting": lag_modell(HistGradientBoostingRegressor(random_state=42)),
}

for navn, modell in modeller.items():
    modell.fit(X_train, y_train)
    resultater.append(evaluer(navn, y_val, modell.predict(X_val)))
    print("Ferdig:", navn)

resultattabell = pd.DataFrame(resultater).sort_values("median_rel_feil")
resultattabell

Ferdig: Ridge
Ferdig: Random Forest
Ferdig: HistGradientBoosting


,modell,median_rel_feil,MAE_mill_eur
3,HistGradientBoosting,0.418417,3.347492
2,Random Forest,0.476159,3.823135
1,Ridge,0.480727,5.531776
0,Baseline (median liga × posisjon),0.700000,5.785878


In [8]:
beste_navn = resultattabell[resultattabell["modell"] != "Baseline (median liga × posisjon)"].iloc[0]["modell"]
beste = modeller[beste_navn]

test_resultat = pd.DataFrame([
    evaluer("Baseline (test)", y_test, baseline_predict(test)),
    evaluer(f"{beste_navn} (test)", y_test, beste.predict(X_test)),
])
test_resultat

,modell,median_rel_feil,MAE_mill_eur
0,Baseline (test),0.700000,5.958220
1,HistGradientBoosting (test),0.421606,3.453338
